# 🚀 Run Gemma on Google Colab GPU & Expose to AI Platform

This notebook installs **Ollama**, loads **Gemma** onto the Colab GPU, and creates a secure **Cloudflare Tunnel** to replace `http://192.168.1.108:11434`.

### Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or any GPU).
2. Run all cells below.
3. Copy the generated `https://xxxx.trycloudflare.com` URL and give it to Antigravity to plug into your AI Platform.

In [ ]:
# Step 1: Install zstd, pciutils, Ollama & Cloudflared Tunnel
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!curl -L -s -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print("✅ Ollama and Cloudflared installed successfully.")

In [ ]:
# Step 2: Start Ollama Server in Background
import subprocess, time, os

# Set OLLAMA_HOST to 0.0.0.0 so tunnel can access it
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Launch ollama serve in background
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("✅ Ollama daemon running on 0.0.0.0:11434")

In [ ]:
# Step 3: Pull Gemma Model into Colab GPU
# Google Colab datacenters download this in ~15-30 seconds
!ollama pull gemma4:latest || ollama pull gemma2:2b || ollama pull gemma:latest
print("✅ Gemma model pulled and verified.")

In [ ]:
# Step 4: Launch Cloudflare Tunnel and Print Public HTTPS URL
import subprocess, re, time

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
print("Starting tunnel, please wait 5-10 seconds...")

for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*65)
        print("🎉 YOUR COLAB GEMMA TUNNEL URL IS READY:")
        print(f"   {tunnel_url}")
        print("="*65 + "\n")
        print(f"OpenAI-Compatible Base URL: {tunnel_url}/v1")
        print("Copy this URL and share it with Antigravity to connect your platform!")
        break

# Keep the tunnel process alive
try:
    cf_proc.wait()
except KeyboardInterrupt:
    print("Tunnel stopped.")